# Module 3: Model Selection, Diagnostics and Honest Uncertainty

*Developed by Yin Zhang, PhD, Assistant Professor, Department of Mathematics and Statistics, Washington State University. Part of the Public Safety Statistics Tutorials, developed for WADEPS through CISER.*

---

Three questions, in the order they have to be answered.

**Which model?** Information criteria narrow the field.
**Is it finished?** Diagnostics decide, and a model that fails them is not a
model whose coefficients you may quote.
**How wrong could it be?** A point forecast is worthless without a
distribution, and a distribution is worthless until someone checks it.

Most analyses do the first, skip the second, and assert the third.

**About 30 minutes.**

## 1. Setup

In [ ]:
# Where the data lives.
#   On Google Colab this reads straight from GitHub.
#   Running from inside a local clone of the repository also works.
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

GITHUB = "https://raw.githubusercontent.com/YinZhangCISER/Public-Safety-Statistics-Tutorials/main/Data/"
_local = Path("../../../Data")
BASE = f"{_local}/" if _local.exists() else GITHUB

monthly = pd.read_csv(BASE + "agency_monthly.csv")
final = monthly[monthly["provisional"] == 0]          # never fit on unfinished months


CALENDAR = pd.period_range("2019-01", "2026-04", freq="M").to_timestamp()


def counts(agency_id):
    """Monthly counts on a complete calendar, so a gap stays visible as missing."""
    d = final[final["agency_id"] == agency_id].sort_values("year_month")
    s = pd.Series(d["n_uof"].values, dtype=float,
                  index=pd.PeriodIndex(d["year_month"], freq="M").to_timestamp())
    s = s.reindex(CALENDAR)
    s.index.freq = "MS"
    return s


def rate(agency_id):
    d = final[final["agency_id"] == agency_id].sort_values("year_month")
    s = pd.Series((100 * d["n_uof"] / d["n_arrests"]).values,
                  index=pd.PeriodIndex(d["year_month"], freq="M").to_timestamp())
    s = s.reindex(CALENDAR)
    s.index.freq = "MS"
    return s


print(f"{final['agency_id'].nunique()} agencies, {final['year_month'].nunique()} months")

from statsmodels.tsa.statespace.sarimax import SARIMAX

s = np.log(counts("A012"))                   # Ashfell, on the log scale
train, test = s.loc[:"2024-12"], s.loc["2025-01":"2025-12"]
print(f"train {len(train)} months, test {len(test)} months")

## 2. Selecting an order

AIC and BIC both trade fit against complexity; BIC penalises extra parameters
harder. Fit the candidates, compare, and **look at whether the two agree**.
When they disagree, the honest report says so rather than picking the flattering
one.

In [ ]:
candidates = [(1, 0, 0, 0, 0, 0), (0, 1, 1, 0, 0, 0), (1, 0, 0, 0, 1, 0),
              (2, 0, 0, 1, 1, 0), (1, 0, 1, 0, 1, 1), (1, 1, 1, 1, 1, 1),
              (0, 1, 1, 0, 1, 1)]

rows = []
for p, d, q, P, D, Q in candidates:
    r = SARIMAX(train, order=(p, d, q), seasonal_order=(P, D, Q, 12),
                enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
    rows.append({"order": f"({p},{d},{q})({P},{D},{Q})12",
                 "AIC": round(r.aic, 1), "BIC": round(r.bic, 1)})

sel = pd.DataFrame(rows).sort_values("AIC").set_index("order")
sel

In [ ]:
print("lowest AIC:", sel["AIC"].idxmin())
print("lowest BIC:", sel["BIC"].idxmin())

Both pick `(0,1,1)(0,1,1)12`, the airline model. Agreement between the two is
worth noting in a report: it means the choice is not sensitive to how hard you
penalise complexity.

**What information criteria cannot do.** They compare models fitted to the
**same data on the same scale**. An AIC from a model on logs cannot be compared
against one on counts, and neither can be compared against a model with a
different differencing order unless the effective sample is the same. They also
say nothing about whether the winner is any good, only that it is better than
the others you tried.

## 3. Diagnostics, which decide whether you may quote anything

In [ ]:
from statsmodels.stats.diagnostic import acorr_ljungbox
from scipy import stats

fit = SARIMAX(train, order=(0, 1, 1), seasonal_order=(0, 1, 1, 12)).fit(disp=False)
res = fit.resid[13:]                         # drop the differencing burn in

checks = {
    "structure left in the residuals (Ljung Box, lag 12)":
        acorr_ljungbox(res, lags=[12], return_df=True)["lb_pvalue"].iloc[0],
    "residuals not normal (Jarque Bera)": stats.jarque_bera(res)[1],
    "changing variance (Ljung Box on squared residuals)":
        acorr_ljungbox(res ** 2, lags=[12], return_df=True)["lb_pvalue"].iloc[0],
}
for name, p in checks.items():
    print(f"  p = {p:.3f}   {'pass' if p > 0.05 else 'FAIL'}   {name}")
print(f"\n  largest standardised residual: {np.abs(res / res.std()).max():.2f}"
      "   (above about 3 deserves a look)")

All three pass and no residual is extreme, so this model has nothing obvious
left in it.

**Each test looks for one thing, and passing all of them is not a guarantee.**
Intermediate [Module 10](../../Intermediate/Notebooks/Module_10_Reading_Autocorrelation.ipynb)
showed a case where Ljung Box is silent about an enormous outlier, because an
outlier is not a correlation. Run the battery, and also look at the residual
plot.

## 4. A forecast distribution, not a forecast

`get_forecast` gives an analytic interval. Simulating from the fitted model
gives the whole distribution. For a well specified model with Gaussian errors
the two agree, and it is worth confirming that rather than assuming it.

In [ ]:
analytic = fit.get_forecast(12)
ci = analytic.conf_int(alpha=0.05)
sim = np.asarray(fit.simulate(12, repetitions=6000, anchor="end",
                              random_state=1)).reshape(12, -1)

a_lo, a_hi = ci.iloc[:, 0].values, ci.iloc[:, 1].values
s_lo, s_hi = np.percentile(sim, 2.5, axis=1), np.percentile(sim, 97.5, axis=1)

print(f"analytic  mean width {np.mean(np.exp(a_hi) - np.exp(a_lo)):.1f} incidents")
print(f"simulated mean width {np.mean(np.exp(s_hi) - np.exp(s_lo)):.1f} incidents")

Essentially identical, which is the expected result and a useful check.

So what does simulation buy? **The back transform.** The model works on logs
and people want counts, and exponentiating a mean is not the mean of the
exponential.

In [ ]:
mu_log = analytic.predicted_mean.values
for i in [0, 5, 11]:
    median = np.exp(mu_log[i])
    mean = np.exp(sim[i]).mean()
    print(f"  month {i + 1:2d}:  exp of the mean of logs {median:6.1f}   "
          f"mean of the simulated counts {mean:6.1f}   difference {100 * (mean / median - 1):+.1f}%")

Exponentiating the forecast of the logs gives the **median** number of
incidents. The **mean** is about two percent higher. Small, and it is the
difference between answering "a typical month" and "how many in total to
expect", which are different questions a chief might ask in the same meeting.

Exponentiating the interval endpoints is fine, because quantiles survive a
monotone transform. Exponentiating the point forecast and calling it the
expected count is not.

## 5. Is the distribution any good? Check it over many origins

Twelve months cannot tell 95 percent coverage from 100 percent. Score the
forecasts at several starting points and pool.

In [ ]:
hits = {0.50: [], 0.80: [], 0.95: []}
pit = []

for end in ["2022-12", "2023-06", "2023-12", "2024-06", "2024-12"]:
    tr = s.loc[:end]
    te = s.loc[pd.Timestamp(end) + pd.offsets.MonthBegin(1):][:12]
    if len(te) < 12:
        continue
    r = SARIMAX(tr, order=(0, 1, 1), seasonal_order=(0, 1, 1, 12)).fit(disp=False)
    draws = np.asarray(r.simulate(12, repetitions=4000, anchor="end",
                                  random_state=7)).reshape(12, -1)
    for level in hits:
        lo = np.percentile(draws, 100 * (1 - level) / 2, axis=1)
        hi = np.percentile(draws, 100 * (1 + level) / 2, axis=1)
        hits[level] += list((te.values >= lo) & (te.values <= hi))
    pit += [float(np.mean(draws[i] <= te.values[i])) for i in range(12)]

pd.DataFrame([{"nominal coverage": f"{int(100 * k)}%",
               "observed": f"{100 * np.mean(v):.1f}%",
               "months": len(v)} for k, v in sorted(hits.items())]).set_index("nominal coverage")

Over 60 forecast months, the 95 percent interval covered 98.3 percent and the
80 percent covered 86.7. **The intervals are wider than they claim**, which is
the opposite of the usual warning and the same shape of surprise as
Intermediate Module 10's robust standard errors.

The 50 percent interval covers 45, slightly narrow, so the miscalibration is
not a simple scaling.

In [ ]:
pit = np.array(pit)
print(f"where the outcome fell in the forecast distribution, over {len(pit)} months")
print(f"  below the 10th percentile: {100 * np.mean(pit < 0.1):.0f}%   should be 10")
print(f"  in the middle 80 percent : {100 * np.mean((pit >= 0.1) & (pit <= 0.9)):.0f}%   should be 80")
print(f"  above the 90th percentile: {100 * np.mean(pit > 0.9):.0f}%   should be 10")

Too few outcomes in the tails. A calibrated forecast would put 10 percent
beyond each edge; this one puts 7 in each. The distribution is a little too
wide, consistently.

**This is a finding you can only get by checking.** Nothing in the model output
hints at it, and a report that quoted the 95 percent interval without this
check would be overstating its own uncertainty by a small but real amount.

## 6. Scoring the whole distribution

Intermediate [Module 15](../../Intermediate/Notebooks/Module_15_Measuring_Forecast_Error.ipynb)
scored point forecasts with MAE and MASE. Those ignore the interval entirely: a
confident wrong forecast and a vague wrong forecast score the same.

**Proper scores** grade the whole distribution. Lower is better, and a forecast
cannot improve its score by misrepresenting its own uncertainty.

In [ ]:
def crps(sample, y):
    """Continuous ranked probability score from a sample of the forecast."""
    sample = np.sort(np.asarray(sample))
    return (np.mean(np.abs(sample - y))
            - 0.5 * np.mean(np.abs(sample[:, None] - sample[None, :])))


def pinball(sample, y, qs=np.arange(0.05, 1.0, 0.05)):
    """Average quantile loss across the distribution."""
    vals = np.quantile(sample, qs)
    return float(np.mean([q * (y - v) if y >= v else (1 - q) * (v - y)
                          for q, v in zip(qs, vals)]))

In [ ]:
rng = np.random.default_rng(0)
resid_sd = float(fit.resid[13:].std())
baseline = np.array([np.exp(train.iloc[-12 + i]) *
                     np.exp(rng.normal(0, resid_sd, 6000)) for i in range(12)])

model_draws = np.exp(sim)
actual = np.exp(test.values)

print(f"{'':22s} {'CRPS':>8s} {'pinball':>9s}")
for name, draws in [("SARIMA", model_draws), ("same month last year", baseline)]:
    c = np.mean([crps(draws[i], actual[i]) for i in range(12)])
    p = np.mean([pinball(draws[i], actual[i]) for i in range(12)])
    print(f"  {name:20s} {c:8.2f} {p:9.2f}")

SARIMA wins on both, by about a third. That is a stronger statement than
beating the baseline on MAE, because it says the model's **whole
distribution** is better, not just its central guess.

## 7. What these intervals still leave out

Every interval above holds the estimated parameters fixed, as though the
model's coefficients were known rather than estimated from 72 months.

The obvious repair is to resample the parameters and simulate again. **For
ARIMA that does not work naively**, and it is worth knowing why before you try
it. The moving average coefficients of the airline model sit close to the
invertibility boundary, so drawing them from a normal approximation puts most
draws outside the region where the model is even defined.

In [ ]:
rng = np.random.default_rng(0)
draws_par = rng.multivariate_normal(fit.params.values, fit.cov_params().values, size=600)
usable = ((np.abs(draws_par[:, 0]) < 1) & (np.abs(draws_par[:, 1]) < 1)
          & (draws_par[:, 2] > 0))
print(f"parameter sets drawn: {len(draws_par)}")
print(f"sets that are actually invertible: {int(usable.sum())}")

Eight of six hundred. An interval built from those eight would be nonsense, and
nothing in the code would warn you.

Doing it properly needs a residual bootstrap with care about the differencing
burn in, or a model fitted in a Bayesian framework where the parameter
uncertainty comes out of the posterior for free. **Both are beyond this
module**, and the honest position in the meantime is to say that the published
intervals condition on the fitted parameters and are therefore somewhat too
narrow on that account, while the calibration check in section 5 found them
somewhat too wide overall.

Those two work in opposite directions. Which dominates is an empirical question
and section 5 is how you answer it.

## 8. What to carry away

| Step | The habit |
|---|---|
| Selection | fit several, report whether AIC and BIC agree |
| Diagnostics | run all three, and look at the residuals as well |
| Point forecast | say whether it is a mean or a median |
| Intervals | simulate, then **check the coverage over many origins** |
| Scoring | MAE for readers, CRPS or pinball when the distribution matters |
| Honesty | state what the interval conditions on |

## Exercise

Run the whole sequence on Stonewick: select an order, check the diagnostics,
and see whether the chosen model passes.

In [ ]:
# Fill in the blank, then run.
AGENCY = None              # try "A001"

if AGENCY:
    x = np.log(counts(AGENCY))
    tr = x.loc[:"2024-12"]
    out = []
    for p, d, q, P, D, Q in [(0, 1, 1, 0, 1, 1), (1, 0, 0, 0, 1, 0),
                             (1, 0, 1, 0, 1, 1), (1, 1, 1, 1, 1, 1)]:
        r = SARIMAX(tr, order=(p, d, q), seasonal_order=(P, D, Q, 12),
                    enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
        rr = r.resid[13:]
        out.append({"order": f"({p},{d},{q})({P},{D},{Q})12",
                    "AIC": round(r.aic, 1), "BIC": round(r.bic, 1),
                    "Ljung Box p": round(acorr_ljungbox(rr, lags=[12],
                                         return_df=True)["lb_pvalue"].iloc[0], 3)})
    print(pd.DataFrame(out).sort_values("AIC").to_string(index=False))
else:
    print("Set AGENCY above, then run this cell again.")

<details>
<summary><b>Solution</b></summary>

```python
AGENCY = "A001"
```

Check two things in the output, in this order.

**Does the AIC winner pass Ljung Box?** If it does not, the lowest AIC is the
best of a set of inadequate models, and reporting its coefficients would be
reporting a model that still has structure in its residuals. Information
criteria rank; they do not certify.

**Do AIC and BIC agree?** When they do not, BIC is preferring the simpler
model and the difference is usually one term. Say which you used and why, and
if the two models give materially different forecasts, report both.

Stonewick adopted the de escalation programme in July 2023, so whatever order
wins here is being asked to describe two regimes with one set of parameters.
That is the subject of [Module 11](Module_11_Interrupted_Time_Series.ipynb),
and it is a good reason not to treat a passing diagnostic as the end of the
matter.

</details>

---

**Next:** Part II, beginning with Module 4, which puts the whole ARIMA
workflow together on one agency.

*Part of the Public Safety Statistics Tutorials, developed for the Washington
Data Exchange for Public Safety (WADEPS) through CISER at Washington State
University. Questions or corrections: yin.zhang@wsu.edu*